In [5]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [6]:
len(documents)

72

In [25]:
documents[1]

{'content': '# Environment\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=3U4gBrmkZyM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nFor this module, all you need is Python with Jupyter.\n\n## Prerequisites\n\nYou need the following:\n\n- Python (3.14 or later)\n- An [OpenAI account](https://openai.com/) (or an OpenAI-compatible\n  provider like Groq, Gemini, or Ollama)\n- Basic familiarity with Python and the command line\n\n## Creating the project\n\nWe\'ll start from scratch - no cloning needed. You\'ll create the\nproject yourself, step by step.\n\nFirst, install uv. It\'s a Python package manager, and I switched all my\nprojects to it because it\'s fast and convenient. Once I started using\nit, I never wanted to go back.\n\nOn Mac or Linux:\n\n```bash\ncurl -LsSf https://astral.sh/uv/install.sh | sh\n```\n\nOn Windows:\n\n```powershell\npowershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"\n```\n\n(You can also use `pip install uv` if you p

In [20]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [21]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [26]:
from evaluation_utils import llm_structured_retry, llm_structured
import json

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "filename": doc["filename"]
        })

    return results, usage

In [29]:
print([doc['filename'] for doc in documents[:3]])

['01-agentic-rag/lessons/01-intro.md', '01-agentic-rag/lessons/02-environment.md', '01-agentic-rag/lessons/03-rag.md']


## Q1. Generating questions

In [30]:
from tqdm.auto import tqdm
import json

ground_truth = []
usages = []

for doc in tqdm(documents[:3]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/3 [00:00<?, ?it/s]

In [31]:
usages

[ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=101, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1121),
 ResponseUsage(input_tokens=1286, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=117, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1403),
 ResponseUsage(input_tokens=1753, input_tokens_details=InputTokensDetails(cached_tokens=1280, cache_write_tokens=0), output_tokens=105, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1858)]

What's the average number of input tokens across these 3 calls?

140
1400
14000
140000

### Answer: 1400

## The full ground truth

In [ ]:
! PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main
! wget ${PREFIX}/cohorts/2026/04-evaluation/ground-truth.csv

## Searching the chunks

In [7]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [22]:
len(chunks)

295

In [27]:
from minsearch import Index, VectorSearch
from embedder import Embedder

emb_model = Embedder()

index = Index(text_fields=["content"])
index.fit(chunks)

X = emb_model.encode_batch([chunk['content'] for chunk in chunks])
vs = VectorSearch()
vs.fit(X, chunks)

def text_search(q, num_results=5):
    index_results = index.search(q, num_results=num_results)
    return index_results

def vector_search(q, num_results=5):
    vs_results = vs.search(emb_model.encode(q), num_results=num_results)
    return vs_results

In [28]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [97]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

## Q2. First result with text search


In [30]:
import pandas as pd
ground_truth = pd.read_csv('ground-truth.csv')
ground_truth = ground_truth.to_dict('index')

In [31]:
ground_truth

{0: {'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 1: {'question': 'Why does this course build the RAG project in plain Python instead of starting with a framework or library?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 2: {'question': 'What are the main weaknesses of large language models that this module is trying to work around?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 3: {'question': 'What will the course build in the first part of the module, and how is the second part different?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 4: {'question': 'What kind of example app are you building here, and what data will it answer questions from?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 5: {'question': 'What do I need installed before starting this module?',
  'filename': '01-agentic-rag/les

In [32]:
q = ground_truth[0]["question"]
q

"What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?"

In [36]:
ground_truth[0]['filename']

'01-agentic-rag/lessons/01-intro.md'

In [34]:
text_search_results = text_search(q)
[item['filename'] for item in text_search_results]

['01-agentic-rag/lessons/03-rag.md',
 '01-agentic-rag/lessons/13-function-calling.md',
 '01-agentic-rag/lessons/03-rag.md',
 '01-agentic-rag/lessons/13-function-calling.md',
 '01-agentic-rag/lessons/01-intro.md']

### Answer: 01-agentic-rag/lessons/03-rag.md

## Q3. First result with vector search


After running vector_search for the same question, what's the filename of the first result?

In [35]:
vector_search_results = vector_search(q)
[item['filename'] for item in vector_search_results]

['01-agentic-rag/lessons/01-intro.md',
 '04-evaluation/lessons/11-evaluation-intro.md',
 '04-evaluation/lessons/12-rag-answers.md',
 '01-agentic-rag/lessons/10-rag-next-steps.md',
 '06-best-practices/lessons/01-intro.md']

### Answer: 01-agentic-rag/lessons/01-intro.md

## Evaluation metrics

In [98]:
def compute_relevance(q, search_function, k=5):
    doc_id = q["filename"]
    try:
        results = search_function(q=q["question"])
    except:
        results = search_function(query=q["question"], k=k)

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

def compute_relevance_total(ground_truth, search_function, k=5):
    relevance_total = []

    for _, q in tqdm(ground_truth.items()):
        relevance = compute_relevance(q, search_function, k)
        relevance_total.append(relevance)

    return relevance_total

In [99]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [100]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [105]:
def evaluate(ground_truth, search_function, k=5):
    relevance_total = compute_relevance_total(ground_truth, search_function, k=k)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

## Q4. Evaluating text search

Evaluate text_search on the ground truth data.

What's the Hit Rate?

In [102]:
from tqdm import tqdm

In [103]:
evaluate(ground_truth, text_search)

100%|██████████| 360/360 [00:00<00:00, 470.23it/s]


{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

### Answer: 0.76

## Q5. Evaluating vector search

Now evaluate vector_search - the part we left for the homework, since the module only evaluated keyword search.

What's the MRR?

In [96]:
evaluate(ground_truth, vector_search)

100%|██████████| 360/360 [00:01<00:00, 197.42it/s]


{'hit_rate': 0.725, 'mrr': 0.5486111111111112}

### Answer: 0.55

## Q6. Tuning hybrid search

In [106]:
for k in [1, 50, 100, 200]:
    print('k=', k, evaluate(ground_truth, hybrid_search, k))

100%|██████████| 360/360 [00:03<00:00, 118.19it/s]


k= 1 {'hit_rate': 0.8388888888888889, 'mrr': 0.6481944444444449}


100%|██████████| 360/360 [00:03<00:00, 117.35it/s]


k= 50 {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}


100%|██████████| 360/360 [00:03<00:00, 119.87it/s]


k= 100 {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}


100%|██████████| 360/360 [00:03<00:00, 117.78it/s]

k= 200 {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}


### Answer: k=1